In [ ]:
# 0,1: 00 for psi_1, 01 for psi_2 ...
# 2,3: For states
# 4,5: Ancillary qubits

In [ ]:
from qiskit import QuantumCircuit
from numpy import sqrt
from qiskit.circuit.library import StatePreparation
def circuit_init():
    qc = QuantumCircuit(6)
    return qc

def PREP(qc):
    desired_vector = [
        1/sqrt(3), 0, 0, 0, 
        1/(4*sqrt(3)), 1/4, 1/4, sqrt(3)/4, 
        1/(4*sqrt(3)), -1/4, -1/4, sqrt(3)/4, 
        0, 0, 0, 0
    ]
    prep = StatePreparation(desired_vector)
    qc.append(prep,[3,2,1,0])
    return qc

In [ ]:
from qiskit.circuit.library.standard_gates import RXGate,RYGate, RZGate
from qiskit.circuit.library import MCXGate
from qiskit.circuit.library.standard_gates import XGate
from pennylane.templates.state_preparations.mottonen import compute_theta, gray_code
from numpy import array, log2

def RR_X(qc, wires, params):
    qc.append(RXGate(params[0]),[wires[0]])
    qc.append(RXGate(params[1]),[wires[1]])

def RR_Z(qc, wires, params):
    qc.append(RZGate(params[0]),[wires[0]])
    qc.append(RZGate(params[1]),[wires[1]])


def CRR_X(qc, wires, state, params):
    qc.append(RXGate(params[0]).control(1,ctrl_state = state[::-1]), [wires[0], wires[1]])  # Control: wires[0] Target: wires[1]
    qc.append(RXGate(params[1]).control(1,ctrl_state = state[::-1]), [wires[0], wires[2]])  # Control: wires[0] Target: wires[2]
    return qc

def CRR_Z(qc, wires, state, params):
    qc.append(RZGate(params[0]).control(1,ctrl_state = state[::-1]), [wires[0], wires[1]])  # Control: wires[0] Target: wires[1]
    qc.append(RZGate(params[1]).control(1,ctrl_state = state[::-1]), [wires[0], wires[2]])  # Control: wires[0] Target: wires[2]
    return qc

def CCRR_X(qc, wires, state, params):
    qc.append(RXGate(params[0]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[2]])  # Control: wires[0] wires[1]
    qc.append(RXGate(params[1]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[3]])  # Control: wires[0] wires[1]
    return qc

def CCRR_Z(qc, wires, state, params):
    qc.append(RZGate(params[0]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[2]])  # Control: wires[0] wires[1]
    qc.append(RZGate(params[1]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[3]])  # Control: wires[0] wires[1]
    return qc

def U_CCR(qc, wires, params):
    qc.append(RYGate(params[0]).control(2, ctrl_state = '00'), [wires[0], wires[1],wires[2]])
    qc.append(RYGate(params[1]).control(2, ctrl_state = '10'), [wires[0], wires[1],wires[2]])
    qc.append(RYGate(params[2]).control(2, ctrl_state = '01'), [wires[0], wires[1],wires[2]])
    qc.append(RYGate(params[3]).control(2, ctrl_state = '11'), [wires[0], wires[1],wires[2]])

def U_CCcR(qc,wires, params):
    qc.append(RYGate(params[0]).control(3, ctrl_state = '001'[::-1]), [wires[0], wires[1],wires[2],wires[3]])
    qc.append(RYGate(params[1]).control(3, ctrl_state = '011'[::-1]), [wires[0], wires[1],wires[2],wires[3]])
    qc.append(RYGate(params[2]).control(3, ctrl_state = '101'[::-1]), [wires[0], wires[1],wires[2],wires[3]])
    qc.append(RYGate(params[3]).control(3, ctrl_state = '111'[::-1]), [wires[0], wires[1],wires[2],wires[3]])

def U_CCR_decom(qc,wires,params):
    # Make sure that control qubits should be increasing order
    params = array(params)
    params = compute_theta(params) #can be on/off

    num_Ucontrols = len(wires)-1
    code = gray_code(2)
    n_selections = len(code)
    control_order = [int(log2(int(code[i], 2) ^ int(code[(i + 1) % n_selections], 2))) for i in range(n_selections)]

    for i in range(n_selections):
        qc.ry(params[i],wires[2])
        qc.cx(wires[num_Ucontrols-1 - control_order[i]],wires[2])
    
    return qc

def U_CCcR_decom(qc,wires,params):
    # Make sure that control qubits should be increasing order
    params = array(params)
    params = compute_theta(params) #can be on/off

    num_Ucontrols = len(wires)-2
    code = gray_code(2)
    n_selections = len(code)
    control_order = [int(log2(int(code[i], 2) ^ int(code[(i + 1) % n_selections], 2))) for i in range(n_selections)]

    for i in range(n_selections):
        qc.cry(params[i],wires[2],wires[3])
        CCX(qc,[wires[num_Ucontrols-1 - control_order[i]],wires[2],wires[3]],'11')
    
    return qc

def CCX(qc, wires, state):
    mcx_gate = MCXGate(num_ctrl_qubits=2,ctrl_state=state[::-1])
    qc.append(mcx_gate, [wires[0], wires[1],wires[2]])
    return qc

def CCCX(qc, wires, state):
    mcx_gate = MCXGate(num_ctrl_qubits=3,ctrl_state=state[::-1])
    qc.append(mcx_gate, [wires[0], wires[1],wires[2], wires[3]])
    return qc

def CX01(qc, wires):
    qc.append(XGate().control(1, ctrl_state='1'),[wires[0], wires[1]])
    return qc


In [ ]:
from qiskit.circuit.library.standard_gates import RYGate
from qiskit.circuit.library import U2Gate
from numpy import pi

# U2Gate 
def Ansatz(qc, wires, params):


    #Big made by HEA 
    RR_X(qc,wires[0:2],params[0:2])
    RR_Z(qc,wires[0:2],params[2:4])
    qc.cx(wires[0],wires[1])
    
    #Module 1
    U_CCR_decom(qc,wires[0:3],params[4:8])
    CRR_X(qc,[wires[2],wires[0],wires[1]],'0',params[8:10])
    CRR_Z(qc,[wires[2],wires[0],wires[1]],'0',params[10:12])
    CCX(qc,[wires[0],wires[2], wires[1]],'10')
    CRR_X(qc,[wires[2],wires[0],wires[1]],'1',params[12:14])
    CRR_Z(qc,[wires[2],wires[0],wires[1]],'1',params[14:16])
    CCX(qc,[wires[0],wires[2], wires[1]],'11')

    #Module 2
    U_CCcR_decom(qc, wires[0:4],params[16:20])
    CCRR_X(qc,[wires[2],wires[3],wires[0],wires[1]],'10',params[20:22])
    CCRR_Z(qc,[wires[2],wires[3],wires[0],wires[1]],'10',params[22:24])
    CCCX(qc,[wires[0],wires[2],wires[3],wires[1]],'110')
    CCRR_X(qc,[wires[2],wires[3],wires[0],wires[1]],'11',params[24:26])
    CCRR_Z(qc,[wires[2],wires[3],wires[0],wires[1]],'11',params[26:28])
    CCCX(qc,[wires[0],wires[2],wires[3],wires[1]],'111')
    
    CX01(qc, [wires[3],wires[2]])
    qc.measure_all()
    return qc

In [ ]:
from numpy.random import rand
num_params=28
params = list(range(num_params))
qc = QuantumCircuit(4)
#PREP(qc)
Ansatz(qc, [0,1,2,3] , params)
qc.draw(output='mpl', style = 'clifford') 

In [ ]:
# ============================================================
# AWS Braket / IQM Garnet setup
# ============================================================

from braket.aws import AwsDevice
from qiskit_braket_provider import BraketProvider
from qiskit import transpile

GARNET_ARN = "arn:aws:braket:eu-north-1::device/qpu/iqm/Garnet"

# Device information only -- does not submit a paid task
garnet_device = AwsDevice(GARNET_ARN)

print("Device name   :", garnet_device.name)
print("Device status :", garnet_device.status)

try:
    qd = garnet_device.queue_depth()
    print("Quantum-task queue :", qd.quantum_tasks)
    print("Hybrid-job queue   :", qd.jobs)
except Exception as exc:
    print("Queue depth unavailable:", exc)

provider = BraketProvider()
garnet_backend = provider.get_backend("Garnet")

print("Backend:", garnet_backend)

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile
import numpy as np
# ============================================================
# Fixed optimal parameters
# ============================================================

optimal_params = np.array([
  3.1807115524e+00, -3.8132930503e-02,  3.5356089616e-02,
  1.5572817653e+00,  3.1455458370e+00,  3.1768345392e+00,
  3.2325289009e+00,  1.3509825044e-02,  1.5108331308e+00,
  4.4207557085e-01,  1.0825267938e+00,  4.0448949435e-01,
  7.7289011723e-01,  1.5579752468e+00,  5.1151626209e-01,
  1.2823600766e+00,  3.1332098408e+00,  1.1224811965e-04,
  3.1150567376e+00, -5.9647728995e-02,  2.2847196429e+00,
  1.6487803630e+00,  2.4156639690e-01, -1.7292916614e-01,
  1.0850155508e+00,  3.2479140653e-01,  6.6764080182e-01,
 -9.8528885264e-01]
, dtype=float)

assert len(optimal_params) == 28

print("Number of optimal parameters:", len(optimal_params))

# ============================================================
# Debug with AerSimulator
# ============================================================

debug_shots = 100000
debug_seed = 150

aersim = AerSimulator()


# ------------------------------------------------------------
# Construct circuit with FIXED optimal parameters
# ------------------------------------------------------------

qc_debug = circuit_init()

PREP(qc_debug)

Ansatz(
    qc_debug,
    [2, 3, 4, 5],
    optimal_params
)

qc_debug = qc_debug.reverse_bits()


# ------------------------------------------------------------
# Transpile for AerSimulator
# ------------------------------------------------------------

t_qc_debug = transpile(
    qc_debug,
    backend=aersim,
    optimization_level=3,
    seed_transpiler=debug_seed
)


print("=== AerSimulator debug ===")
print("Depth :", t_qc_debug.depth())
print("Size  :", t_qc_debug.size())
print("Ops   :", t_qc_debug.count_ops())


# ------------------------------------------------------------
# Run simulator
# ------------------------------------------------------------

job_debug = aersim.run(
    t_qc_debug,
    shots=debug_shots,
    seed_simulator=debug_seed
)

result_debug = job_debug.result()

counts_debug = result_debug.get_counts()


print("\nCounts:")
print(counts_debug)


# ------------------------------------------------------------
# Same target-count mapping as existing notebook
# ------------------------------------------------------------

tar_00 = 0
tar_01 = 0
tar_10 = 0

for outcome, count in counts_debug.items():

    if outcome[0:2] == "00":
        if outcome[4:6] == "00":
            tar_00 += count

    if outcome[0:2] == "01":
        if outcome[4:6] == "01":
            tar_01 += count

    if outcome[0:2] == "10":
        if outcome[4:6] == "10":
            tar_10 += count


p_suc_debug = (
    tar_00
    + tar_01
    + tar_10
) / debug_shots


print("\n=== Debug result ===")
print("tar_00 :", tar_00)
print("tar_01 :", tar_01)
print("tar_10 :", tar_10)

print(
    "Success probability :",
    p_suc_debug
)

In [ ]:
# ============================================================
# IQM Garnet: one run with optimal parameters
# ============================================================

import time

num_shots = 10000
seed_transpiler = 150


# ------------------------------------------------------------
# Construct actual circuit
# ------------------------------------------------------------

qc = circuit_init()

PREP(qc)

Ansatz(
    qc,
    [2, 3, 4, 5],
    optimal_params
)

qc_reverse = qc.reverse_bits()


# ------------------------------------------------------------
# Transpile for IQM Garnet
# ------------------------------------------------------------

t_qc = transpile(
    qc_reverse,
    backend=garnet_backend,
    optimization_level=3,
    seed_transpiler=seed_transpiler
)


# ============================================================
# IQM Garnet transpiled resource report
# ============================================================

op_counts = dict(t_qc.count_ops())

one_q = 0
two_q = 0
multi_q = 0
measurements = 0

for item in t_qc.data:

    name = item.operation.name
    nq = len(item.qubits)

    if name == "measure":
        measurements += 1
    elif nq == 1:
        one_q += 1
    elif nq == 2:
        two_q += 1
    elif nq > 2:
        multi_q += 1


print("\n=== IQM Garnet transpiled resource report ===")
print(f"{'num_qubits':22s}: {t_qc.num_qubits}")
print(f"{'depth':22s}: {t_qc.depth()}")
print(f"{'size':22s}: {t_qc.size()}")
print(f"{'1q_gate_count':22s}: {one_q}")
print(f"{'2q_gate_count':22s}: {two_q}")
print(f"{'multiq_gate_count':22s}: {multi_q}")
print(f"{'measurement_count':22s}: {measurements}")
print(f"{'cz_count':22s}: {int(op_counts.get('cz', 0))}")
print(f"{'cx_count':22s}: {int(op_counts.get('cx', 0))}")
print(f"{'operation_counts':22s}: {op_counts}")


# ============================================================
# Submit ONE QPU task
# ============================================================

start_time = time.perf_counter()

job = garnet_backend.run(
    t_qc,
    shots=num_shots,
    verbatim=True
)

print("\nBraket task ID:")
print(job.job_id())

print("\nWaiting for result...")

result = job.result()

end_time = time.perf_counter()

print("Task wall time:", end_time - start_time, "s")


# ------------------------------------------------------------
# Counts
# ------------------------------------------------------------

counts = result.get_counts()

print("\nCounts:")
print(counts)


# ------------------------------------------------------------
# Existing DT target-count mapping
# ------------------------------------------------------------

tar_00 = 0
tar_01 = 0
tar_10 = 0

for outcome, count in counts.items():

    if outcome[0:2] == "00":
        if outcome[4:6] == "00":
            tar_00 += count

    if outcome[0:2] == "01":
        if outcome[4:6] == "01":
            tar_01 += count

    if outcome[0:2] == "10":
        if outcome[4:6] == "10":
            tar_10 += count


p_suc = (
    tar_00
    + tar_01
    + tar_10
) / num_shots


print("\n=== Garnet result ===")
print("tar_00 :", tar_00)
print("tar_01 :", tar_01)
print("tar_10 :", tar_10)
print("Success probability :", p_suc)